In [361]:
import numpy as np
import pandas as pd
import sklearn
import os
import json
from datetime import datetime
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import cross_validate, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (classification_report, roc_auc_score,
                              f1_score, recall_score, confusion_matrix)

from scipy.stats import loguniform # Hyperparameter Tunning

import warnings
warnings.filterwarnings("ignore")

In [322]:
data = pd.read_csv("../data/clean/build_dataset.csv")

In [323]:
#data.info()

In [324]:
#data.columns

In [325]:
X = data.drop("churn_value", axis=1)
y = data["churn_value"]

In [326]:
# stratify -> The churn rate (26%) is maintained in both sets; if stratify is not used, it may randomly split unevenly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [327]:
print(f"Train : {X_train.shape[0]} rows | Test : {X_test.shape[0]} rows")
print(f"Train churn rate : {y_train.mean():.2%}")
print(f"Test  churn rate : {y_test.mean():.2%}\n")

Train : 5634 rows | Test : 1409 rows
Train churn rate : 26.54%
Test  churn rate : 26.54%



In [328]:
numeric_cols = [
    "tenure_months", "monthly_charges", "total_charges", "cltv", "avg_monthly_charge", "charge_vs_avg", "payment_ratio",
    "total_services", "cost_per_service", "high_risk", "engagement_score", "log_total_charges", "log_monthly_charges",
    "log_cltv", "tenure_x_contract", "charge_x_risk",
]

In [329]:
binary_cols = [
    "senior_citizen", "partner", "dependents", "phone_service", "paperless_billing", "online_security", "online_backup",
    "device_protection", "tech_support", "streaming_tv", "streaming_movies",
]

In [330]:
gender_cols = ["gender"]

In [331]:
ordinal_cols        = ["contract", "tenure_group"]
contract_categories = [["Month-to-month", "One year", "Two year"]]
tenure_categories   = [["New", "Growing", "Mature", "Loyal"]]

In [332]:
nominal_cols = ["multiple_lines", "internet_service", "payment_method"]

In [333]:
preprocessor = ColumnTransformer(transformers=[
    ("num",     StandardScaler(),
                numeric_cols),

    ("binary",  OrdinalEncoder(categories=[["No", "Yes"]] * len(binary_cols)),
                binary_cols),

    ("gender",  OrdinalEncoder(categories=[["Female", "Male"]]),
                gender_cols),

    ("ordinal", OrdinalEncoder(categories=contract_categories + tenure_categories),
                ordinal_cols),

    ("nominal", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
                nominal_cols),
],
remainder="drop"
)

In [334]:
cat_features = binary_cols + gender_cols + ordinal_cols + nominal_cols #catboost

In [335]:
#catboost
X_train_raw = X_train.copy()
X_test_raw  = X_test.copy()

In [336]:
X_train_encoded = preprocessor.fit_transform(X_train, y_train)
X_test_encoded = preprocessor.transform(X_test)

In [337]:
X_train_df = pd.DataFrame(X_train_encoded, columns=preprocessor.get_feature_names_out(), index=X_train.index)
X_test_df = pd.DataFrame(X_test_encoded, columns=preprocessor.get_feature_names_out(), index=X_test.index)

In [338]:
X_train_df.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 4626 to 6017
Data columns (total 36 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   num__tenure_months                               5634 non-null   float64
 1   num__monthly_charges                             5634 non-null   float64
 2   num__total_charges                               5634 non-null   float64
 3   num__cltv                                        5634 non-null   float64
 4   num__avg_monthly_charge                          5634 non-null   float64
 5   num__charge_vs_avg                               5634 non-null   float64
 6   num__payment_ratio                               5634 non-null   float64
 7   num__total_services                              5634 non-null   float64
 8   num__cost_per_service                            5634 non-null   float64
 9   num__high_risk                             

In [339]:
def calculate_classification_metrics(y_true, y_pred, y_prob):
    f1 = f1_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)

    return f1, recall, roc_auc, cm

In [340]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

classification_models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    "XGBoost":             XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, verbosity=0, n_jobs=-1),
    "LightGBM":            LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1, n_jobs=-1),
    "CatBoost":            CatBoostClassifier(auto_class_weights="Balanced", random_state=42, verbose=0)
}

In [341]:
model_names = list(classification_models.keys())
results = {}

for name in model_names:
    model = classification_models[name]

    if name == "CatBoost":
        model.fit(X_train_raw, y_train, cat_features=cat_features)

        y_train_pred = model.predict(X_train_raw)
        y_test_pred  = model.predict(X_test_raw)

        y_train_prob = model.predict_proba(X_train_raw)[:, 1]
        y_test_prob  = model.predict_proba(X_test_raw)[:, 1]

    else:
        model.fit(X_train_df, y_train)

        y_train_pred = model.predict(X_train_df)
        y_test_pred = model.predict(X_test_df)

        # Sınıf Olasılık Tahminleri (ROC AUC skoru için 1 olma olasılığı gerekir)
        y_train_prob = model.predict_proba(X_train_df)[:, 1]
        y_test_prob = model.predict_proba(X_test_df)[:, 1]

    train_f1, train_recall, train_roc_auc, train_cm = calculate_classification_metrics(
        y_train, y_train_pred, y_train_prob
    )
    test_f1, test_recall, test_roc_auc, test_cm = calculate_classification_metrics(
        y_test, y_test_pred, y_test_prob
    )

    print(f"=== {name} ===")

    print("Evaluation for Training Set")
    print("F1 Score  :", round(train_f1, 4))
    print("Recall    :", round(train_recall, 4))
    print("ROC AUC   :", round(train_roc_auc, 4))
    print("Confusion Matrix :\n", train_cm)

    print("------------------------")

    print("Evaluation for Test Set")
    print("F1 Score  :", round(test_f1, 4))
    print("Recall    :", round(test_recall, 4))
    print("ROC AUC   :", round(test_roc_auc, 4))
    print("Confusion Matrix :\n", test_cm)

    print("------------------------")
    print(classification_report(y_test, y_test_pred))

    print("\n")

    results[name] = {
        "Train ROC-AUC": round(train_roc_auc, 4),
        "Test ROC-AUC" : round(test_roc_auc, 4),
        "Test F1"      : round(test_f1, 4),
        "Test Recall"  : round(test_recall, 4),
        "Overfit"      : round(train_roc_auc - test_roc_auc, 4),
    }

=== Logistic Regression ===
Evaluation for Training Set
F1 Score  : 0.6547
Recall    : 0.8268
ROC AUC   : 0.8669
Confusion Matrix :
 [[3094 1045]
 [ 259 1236]]
------------------------
Evaluation for Test Set
F1 Score  : 0.6216
Recall    : 0.7861
ROC AUC   : 0.8556
Confusion Matrix :
 [[757 278]
 [ 80 294]]
------------------------
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



=== Random Forest ===
Evaluation for Training Set
F1 Score  : 1.0
Recall    : 1.0
ROC AUC   : 1.0
Confusion Matrix :
 [[4139    0]
 [   0 1495]]
------------------------
Evaluation for Test Set
F1 Score  : 0.5477
Recall    : 0.4759
ROC AUC   : 0.8437
Confusion Matrix :
 [[937  98]
 [196 178]]
------------------------
              prec

In [349]:
results_df = pd.DataFrame(results).T.sort_values("Test ROC-AUC", ascending=False)
print(results_df.to_string())

                     Train ROC-AUC  Test ROC-AUC  Test F1  Test Recall  Overfit
Logistic Regression         0.8669        0.8556   0.6216       0.7861   0.0113
CatBoost                    0.9476        0.8534   0.6467       0.7807   0.0942
LightGBM                    0.9770        0.8488   0.6237       0.7246   0.1282
Random Forest               1.0000        0.8437   0.5477       0.4759   0.1563
XGBoost                     0.9984        0.8345   0.6025       0.6444   0.1639


In [343]:
lr_params = {
    "model__C": loguniform(1e-3, 100),
    "model__solver": ["liblinear", "lbfgs"],
    "model__penalty": ["l2"],
    "model__class_weight": [None, "balanced"]
}

rf_params = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [5, 10, 15, 20,30, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2"],
    "model__bootstrap": [True, False],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

xgb_params = {
    "model__n_estimators": [200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.3],
    "model__min_child_weight": [1, 3, 5]
}

lgbm_params = {
    "model__n_estimators": [200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__num_leaves": [15, 31, 50, 70],
    "model__max_depth": [-1, 5, 10, 15],
    "model__min_child_samples": [10, 20, 30],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__class_weight":[None,"balanced"]
}

cat_params = {
    "iterations": [300, 500, 700],
    "depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
    "bagging_temperature": [0, 1, 3, 5],
    "auto_class_weights":[None, "Balanced"]
}

In [344]:
param_grids = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42), lr_params),
    ("Random Forest",       RandomForestClassifier(random_state=42, n_jobs=-1), rf_params),
    ("XGBoost",             XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1), xgb_params),
    ("LightGBM",            LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1), lgbm_params),
    ("CatBoost",            CatBoostClassifier(verbose=0, random_state=42), cat_params),
]


In [345]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [346]:
"""for name, model, params in param_grids:
    randomcv_model = RandomizedSearchCV(estimator=model, param_distributions=params, n_iter=30, cv=kf, verbose=2, n_jobs=-1, scoring="roc_auc", random_state=42)
    randomcv_model.fit(X_train, y_train)
    print("Best params for: ", name, randomcv_model.best_params_)"""

'for name, model, params in param_grids:\n    randomcv_model = RandomizedSearchCV(estimator=model, param_distributions=params, n_iter=30, cv=kf, verbose=2, n_jobs=-1, scoring="roc_auc", random_state=42)\n    randomcv_model.fit(X_train, y_train)\n    print("Best params for: ", name, randomcv_model.best_params_)'

In [347]:
# The "model__" prefix is required to pass parameters to the model inside the pipeline.
# CatBoost is outside the pipeline, so there is no prefix.

tuning_results  = {}
best_estimators = {}

print("Hyperparameter Tuning")

for name, model, params in param_grids:
    print(f"\n TUNING -> {name}")

    if name == "CatBoost":
        # CatBoost automatically encodes categorical columns
        # No pipeline or preprocessor is used, raw data is provided
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=30,
            cv=kf,
            scoring="roc_auc",
            refit=True,
            n_jobs=-1,
            random_state=42,
            verbose=0,
        )
        search.fit(X_train_raw, y_train, cat_features=cat_features)

    else:
        # Other models: preprocessor inside the pipeline
        # In each fold, the preprocessor is fitted only to the train portion of that fold -> no leakage
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])
        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=params,
            n_iter=30,
            cv=kf,
            scoring="roc_auc",
            refit=True,
            n_jobs=-1,
            random_state=42,
            verbose=0,
        )
        search.fit(X_train, y_train)

    print(f"En iyi parametreler : {search.best_params_}")
    print(f"En iyi CV ROC-AUC   : {search.best_score_:.4f}")

    best_estimator = search.best_estimator_
    best_estimators[name] = best_estimator

    # 5-FOLD CV (with the best parameters, to also see f1 and recall) RandomizedSearchCV already generated the CV, but only optimized roc_auc.
    # Evaluate the tuned model using Stratified K-Fold Cross Validation

    if name == "CatBoost":
        cv_scores = cross_validate(
            best_estimator, X_train_raw, y_train,
            cv=kf,
            scoring=["roc_auc", "f1", "recall"],
            params={"cat_features": cat_features},
            n_jobs=-1,
        )
    else:
        cv_scores = cross_validate(
            best_estimator, X_train, y_train,
            cv=kf,
            scoring=["roc_auc", "f1", "recall"],
            n_jobs=-1,
        )

    cv_roc = cv_scores["test_roc_auc"].mean()
    cv_f1  = cv_scores["test_f1"].mean()
    cv_rec = cv_scores["test_recall"].mean()

    print(f"\n5-Fold CV Sonuclari:")
    print(f"  ROC-AUC : {cv_roc:.4f} +/- {cv_scores['test_roc_auc'].std():.4f}")
    print(f"  F1      : {cv_f1:.4f} +/- {cv_scores['test_f1'].std():.4f}")
    print(f"  Recall  : {cv_rec:.4f} +/- {cv_scores['test_recall'].std():.4f}")

    # Train-Test Final Evaluation

    if name == "CatBoost":
        y_train_pred = best_estimator.predict(X_train_raw)
        y_test_pred  = best_estimator.predict(X_test_raw)

        y_train_prob = best_estimator.predict_proba(X_train_raw)[:, 1]
        y_test_prob  = best_estimator.predict_proba(X_test_raw)[:, 1]
    else:
        y_train_pred = best_estimator.predict(X_train)
        y_test_pred  = best_estimator.predict(X_test)

        y_train_prob = best_estimator.predict_proba(X_train)[:, 1]
        y_test_prob  = best_estimator.predict_proba(X_test)[:, 1]

    train_f1, train_recall, train_roc_auc, train_cm = calculate_classification_metrics(
        y_train, y_train_pred, y_train_prob
    )
    test_f1, test_recall, test_roc_auc, test_cm = calculate_classification_metrics(
        y_test, y_test_pred, y_test_prob
    )

    print(f"\n--- {name} (Tuned) ---")
    print("Evaluation for Training Set")
    print("F1 Score  :", round(train_f1, 4))
    print("Recall    :", round(train_recall, 4))
    print("ROC AUC   :", round(train_roc_auc, 4))
    print("Confusion Matrix :\n", train_cm)

    print("------------------------")

    print("Evaluation for Test Set")
    print("F1 Score  :", round(test_f1, 4))
    print("Recall    :", round(test_recall, 4))
    print("ROC AUC   :", round(test_roc_auc, 4))
    print("Confusion Matrix :\n", test_cm)

    print("------------------------")

    print(classification_report(y_test, y_test_pred))
    print("\n")

    tuning_results[name] = {
        "CV ROC-AUC"   : round(cv_roc, 4),
        "CV F1"        : round(cv_f1, 4),
        "CV Recall"    : round(cv_rec, 4),
        "Train ROC-AUC": round(train_roc_auc, 4),
        "Test ROC-AUC" : round(test_roc_auc, 4),
        "Test F1"      : round(test_f1, 4),
        "Test Recall"  : round(test_recall, 4),
        "Overfit"      : round(train_roc_auc - test_roc_auc, 4),
    }

Hyperparameter Tuning

 TUNING -> Logistic Regression
En iyi parametreler : {'model__C': np.float64(4.5705630998014515), 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
En iyi CV ROC-AUC   : 0.8624

5-Fold CV Sonuclari:
  ROC-AUC : 0.8624 +/- 0.0132
  F1      : 0.6184 +/- 0.0333
  Recall  : 0.5679 +/- 0.0398

--- Logistic Regression (Tuned) ---
Evaluation for Training Set
F1 Score  : 0.6231
Recall    : 0.5679
ROC AUC   : 0.8672
Confusion Matrix :
 [[3758  381]
 [ 646  849]]
------------------------
Evaluation for Test Set
F1 Score  : 0.62
Recall    : 0.5802
ROC AUC   : 0.8545
Confusion Matrix :
 [[926 109]
 [157 217]]
------------------------
              precision    recall  f1-score   support

           0       0.86      0.89      0.87      1035
           1       0.67      0.58      0.62       374

    accuracy                           0.81      1409
   macro avg       0.76      0.74      0.75      1409
weighted avg       0.80      0.81      0.8

In [348]:
print("--- Comparison After Tuning ---")
tuning_df = pd.DataFrame(tuning_results).T.sort_values("Test ROC-AUC", ascending=False)
print(tuning_df.to_string())

--- Comparison After Tuning ---
                     CV ROC-AUC   CV F1  CV Recall  Train ROC-AUC  Test ROC-AUC  Test F1  Test Recall  Overfit
CatBoost                 0.8626  0.6461     0.8033         0.8906        0.8565   0.6412       0.8075   0.0340
Logistic Regression      0.8624  0.6184     0.5679         0.8672        0.8545   0.6200       0.5802   0.0127
LightGBM                 0.8606  0.6089     0.5572         0.9096        0.8540   0.6073       0.5561   0.0556
Random Forest            0.8590  0.6562     0.7552         0.9424        0.8529   0.6408       0.7513   0.0894
XGBoost                  0.8616  0.6451     0.8147         0.8919        0.8519   0.6312       0.8102   0.0400


In [351]:
# Best Model
tuning_df    = pd.DataFrame(tuning_results).T.sort_values("Test ROC-AUC", ascending=False)
best_name    = tuning_df.index[0]
best_model   = best_estimators[best_name]

print("--- Best Model ---")
print(f"Model        : {best_name}")

print(f"CV ROC-AUC   : {tuning_results[best_name]['CV ROC-AUC']}")
print(f"Test ROC-AUC : {tuning_results[best_name]['Test ROC-AUC']}")
print(f"Test F1      : {tuning_results[best_name]['Test F1']}")
print(f"Test Recall  : {tuning_results[best_name]['Test Recall']}")
print(f"Overfit      : {tuning_results[best_name]['Overfit']}")

# Final confusion matrix ve classification report
if best_name == "CatBoost":
    y_final_pred = best_model.predict(X_test_raw)
    y_final_prob = best_model.predict_proba(X_test_raw)[:, 1]
else:
    y_final_pred = best_model.predict(X_test)
    y_final_prob = best_model.predict_proba(X_test)[:, 1]

print("\nFinal Confusion Matrix:")
print(confusion_matrix(y_test, y_final_pred))

print("\nFinal Classification Report:")
print(classification_report(y_test, y_final_pred))

--- Best Model ---
Model        : CatBoost
CV ROC-AUC   : 0.8626
Test ROC-AUC : 0.8565
Test F1      : 0.6412
Test Recall  : 0.8075
Overfit      : 0.034

Final Confusion Matrix:
[[769 266]
 [ 72 302]]

Final Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1035
           1       0.53      0.81      0.64       374

    accuracy                           0.76      1409
   macro avg       0.72      0.78      0.73      1409
weighted avg       0.81      0.76      0.77      1409



In [358]:
#features_importances_
if best_name == "CatBoost":
    importances = best_model.get_feature_importance()

    fi_df = (
        pd.DataFrame({
            "Feature": X_train_raw.columns,
            "Importance": importances
        })
        .sort_values("Importance", ascending=False)
    )

    print(fi_df.head(15))

# tree-based models in feature_importances
elif best_name in ["Random Forest", "XGBoost", "LightGBM"]:

    model_step = best_model.named_steps["model"]
    prep_step = best_model.named_steps["preprocessor"]

    fi_df = (
        pd.DataFrame({
            "Feature": prep_step.get_feature_names_out(),
            "Importance": model_step.feature_importances_
        })
        .sort_values("Importance", ascending=False)
    )

    print(fi_df.head(15))

elif best_name == "Logistic Regression":

    model_step = best_model.named_steps["model"]
    prep_step = best_model.named_steps["preprocessor"]

    coef_df = (
        pd.DataFrame({
            "Feature": prep_step.get_feature_names_out(),
            "Coefficient": model_step.coef_[0]
        })
        # Positive: Increases the risk of churn,
        # Negative: Reduces the risk of churn
        .assign(Abs=lambda x: x["Coefficient"].abs())
        .sort_values("Abs", ascending=False)
        .drop(columns="Abs")
    )

    print(coef_df.head(15))

              Feature  Importance
3          dependents   22.066468
14           contract   11.416615
7    internet_service    8.777739
27   engagement_score    6.337118
28  tenure_x_contract    5.533089
4       tenure_months    4.413881
20       tenure_group    3.724073
16     payment_method    3.132674
25   cost_per_service    2.903573
29      charge_x_risk    2.813522
26          high_risk    2.725487
23      payment_ratio    2.713257
30  log_total_charges    2.505464
18      total_charges    2.283082
19               cltv    1.757727


In [359]:
os.makedirs("../models/saved", exist_ok=True)
saved_paths = {}

print("--- Model Saved")

for name, estimator in best_estimators.items():
    # "Logistic Regression" -> "logistic_regression.joblib"
    file_name        = name.lower().replace(" ", "_")
    file_path        = f"../models/saved/{file_name}.joblib"
    joblib.dump(estimator, file_path)
    saved_paths[name] = file_path
    print(f"Saved : {file_path}")


--- Model Saved
Saved : ../models/saved/logistic_regression.joblib
Saved : ../models/saved/random_forest.joblib
Saved : ../models/saved/xgboost.joblib
Saved : ../models/saved/lightgbm.joblib
Saved : ../models/saved/catboost.joblib


In [360]:
# all models metrics saved model_registry.json
registry = {
    "best_model_key"  : best_name,
    "best_model_path" : saved_paths[best_name],
    "selection_metric": "test_roc_auc",
    "created_at"      : datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
    "models"          : {}
}

for name in best_estimators:
    registry["models"][name] = {
        "path"        : saved_paths[name],
        "cv_roc_auc"  : tuning_results[name]["CV ROC-AUC"],
        "test_roc_auc": tuning_results[name]["Test ROC-AUC"],
        "test_f1"     : tuning_results[name]["Test F1"],
        "test_recall" : tuning_results[name]["Test Recall"],
        "overfit"     : tuning_results[name]["Overfit"],
    }

os.makedirs("../models", exist_ok=True)
registry_path = "../models/model_registry.json"

with open(registry_path, "w") as f:
    json.dump(registry, f, indent=4)

print(f"\nRegistry saved... : {registry_path}")
print(json.dumps(registry, indent=4))



Registry saved... : ../models/model_registry.json
{
    "best_model_key": "CatBoost",
    "best_model_path": "../models/saved/catboost.joblib",
    "selection_metric": "test_roc_auc",
    "created_at": "2026-07-16T17:48:03",
    "models": {
        "Logistic Regression": {
            "path": "../models/saved/logistic_regression.joblib",
            "cv_roc_auc": 0.8624,
            "test_roc_auc": 0.8545,
            "test_f1": 0.62,
            "test_recall": 0.5802,
            "overfit": 0.0127
        },
        "Random Forest": {
            "path": "../models/saved/random_forest.joblib",
            "cv_roc_auc": 0.859,
            "test_roc_auc": 0.8529,
            "test_f1": 0.6408,
            "test_recall": 0.7513,
            "overfit": 0.0894
        },
        "XGBoost": {
            "path": "../models/saved/xgboost.joblib",
            "cv_roc_auc": 0.8616,
            "test_roc_auc": 0.8519,
            "test_f1": 0.6312,
            "test_recall": 0.8102,
          